# Bronze — Statistics Iceland Ingestion
Fetches quarterly GDP YoY growth, House Price Index, and Wage Index from Statistics Iceland (Hagstofa Íslands) via the PX-Web REST API.

**Source:** Hagstofa Íslands PX-Web API  
**Datasets:** THJ01601 (GDP), VIS01106 (HPI), LAU04000 (Wage Index)  
**Output:** `bronze.statistics_iceland_raw` (Delta table) — one row per observation, `dataset` column identifies the series

In [ ]:
import requests
import pandas as pd

REBUILD_HISTORY = True

if spark.catalog.tableExists("bronze.statistics_iceland_raw") and not REBUILD_HISTORY:
    is_full_load = False
    print("Incremental mode.")
else:
    is_full_load = True
    print("Full load mode.")

In [ ]:
# GDP — Quarterly YoY Growth (THJ01601)
GDP_URL = "https://px.hagstofa.is/pxis/api/v1/is/Efnahagur/thjodhagsreikningar/landsframl/2_landsframleidsla_arsfj/THJ01601.px"

resp_gdp = requests.post(GDP_URL, json={
    "query": [
        {"code": "Mælikvarði", "selection": {"filter": "item", "values": ["2"]}},
        {"code": "Skipting",   "selection": {"filter": "item", "values": ["14"]}},
    ],
    "response": {"format": "json"},
})

if resp_gdp.status_code == 400:
    print(f"GDP API Error Detail: {resp_gdp.text}")

resp_gdp.raise_for_status()

gdp_records = [
    {
        "dataset": "gdp", 
        "period_raw": i["key"][2], # Quarter is at index [2]
        "value_raw": float(i["values"][0]), 
        "ingested_at": pd.Timestamp.now()
    }
    for i in resp_gdp.json()["data"]
    if i["values"][0] not in (".", "", None)
]

In [ ]:
# HPI — House Price Index (VIS01106)
HPI_URL = "https://px.hagstofa.is/pxis/api/v1/is/Efnahagur/visitolur/1_vnv/3_greiningarvisitolur/VIS01106.px"

resp_hpi = requests.post(HPI_URL, json={
    "query": [
        {
            "code": "Vísitala", 
            "selection": {"filter": "item", "values": ["RI_total"]}
        }
    ],
    "response": {"format": "json"}
})

if resp_hpi.status_code == 400:
    print(f"HPI API Error Detail: {resp_hpi.text}")

resp_hpi.raise_for_status()

hpi_records = [
    {
        "dataset": "hpi", 
        "period_raw": i["key"][0], # Month is at index [0]
        "value_raw": float(i["values"][0]),
        "ingested_at": pd.Timestamp.now()
    }
    for i in resp_hpi.json()["data"]
    if i["values"][0] not in (".", "", None)
]

In [ ]:
# Wage Index (LAU04000)
WAGE_URL = "https://px.hagstofa.is/pxis/api/v1/is/Samfelag/launogtekjur/2_lvt/1_manadartolur/LAU04000.px"

resp_wages = requests.post(WAGE_URL, json={
    "query": [
        {"code": "Eining", "selection": {"filter": "item", "values": ["index"]}}
    ],
    "response": {"format": "json"}
})

if resp_wages.status_code == 400:
    print(f"Wage API Error Detail: {resp_wages.text}")

resp_wages.raise_for_status()

wage_records = [
    {
        "dataset": "wage_index", 
        "period_raw": i["key"][0], # Month is at index [0]
        "value_raw": float(i["values"][0]), 
        "ingested_at": pd.Timestamp.now()
    }
    for i in resp_wages.json()["data"]
    if i["values"][0] not in (".", "", None)
]

In [ ]:
new_data = pd.DataFrame(gdp_records + hpi_records + wage_records)
spark_df = spark.createDataFrame(new_data)

if is_full_load:
    spark_df.write.format("delta") \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable("bronze.statistics_iceland_raw")
    print(f"Full load: {spark_df.count()} rows")
else:
    spark_df.write.format("delta") \
            .mode("append") \
            .option("mergeSchema", "true") \
            .saveAsTable("bronze.statistics_iceland_raw")
    print(f"Appended {spark_df.count()} rows")